# Train And Benchmark

Thin orchestration notebook for `debug_tiny`, `debug_synthetic_generalization`, `debug_synthetic_generalization_large`, `local_small`, and `hg002_chr20_small`.
All project logic is run through the pinned conda interpreter via subprocess so the notebook itself stays import-light.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from pprint import pprint

ABSOLUTE_PROJECT_FALLBACK = Path('/Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild')

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'scripts' / 'train_model.py').exists() and (candidate / 'configs' / 'debug_tiny.yaml').exists():
            return candidate
    if (ABSOLUTE_PROJECT_FALLBACK / 'scripts' / 'train_model.py').exists():
        return ABSOLUTE_PROJECT_FALLBACK
    raise FileNotFoundError('Could not locate omega_lr_rebuild project root from notebook cwd.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PYTHON = '/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python'
PRESET = 'debug_synthetic_generalization'  # or 'debug_tiny', 'debug_synthetic_generalization_large', 'local_small', 'hg002_chr20_small'
RUN_FETCH = False

# Primary surgical debug path.
RUN_NAME = 'full'
ENABLE_LOW_THRESHOLD_DEBUG = True
ENABLE_ARGMAX_DEBUG = True

# Mandatory rescue path: prove staged overfit before treating benchmarks as meaningful.
ENABLE_OVERFIT_DEBUG = True
OVERFIT_RUN_NAME = 'full'  # target_only or full
OVERFIT_NUM_EXAMPLES = 4
OVERFIT_CASES = ['sub', 'del', 'ins', 'copy']

CONFIG_PATH = PROJECT_ROOT / 'configs' / f'{PRESET}.yaml'
TEMP_CONFIG_PATH = PROJECT_ROOT / 'outputs' / 'tmp' / f'{PRESET}_active_debug.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('sys.executable =', sys.executable)
print('PYTHON =', PYTHON)


In [ ]:
def run_script(*args, check=True):
    command = [PYTHON, *args]
    print(' '.join(str(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env={**os.environ, 'PYTHONPATH': str(PROJECT_ROOT / 'src')},
        check=False,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise subprocess.CalledProcessError(completed.returncode, command, completed.stdout, completed.stderr)
    return completed


if PRESET == 'debug_tiny':
    print('Running mandatory staged overfit suite before benchmark/debug runs...')
    run_script('scripts/run_overfit_suite.py', '--output-dir', 'outputs/mandatory_overfit')
    OVERFIT_SUITE_PATH = PROJECT_ROOT / 'outputs' / 'mandatory_overfit' / 'overfit_suite.json'
    pprint(json.loads(OVERFIT_SUITE_PATH.read_text(encoding='utf-8')))


make_config_args = [
    'scripts/make_debug_config.py',
    '--base-config', str(CONFIG_PATH),
    '--output-config', str(TEMP_CONFIG_PATH),
    '--preset', PRESET,
    '--run-name', RUN_NAME,
]
if ENABLE_LOW_THRESHOLD_DEBUG:
    make_config_args.append('--enable-low-threshold-debug')
if ENABLE_OVERFIT_DEBUG and PRESET == 'debug_tiny':
    make_config_args.extend([
        '--enable-overfit-debug',
        '--overfit-run-name', OVERFIT_RUN_NAME,
        '--overfit-num-examples', str(OVERFIT_NUM_EXAMPLES),
        '--overfit-cases', ','.join(OVERFIT_CASES),
    ])
elif ENABLE_OVERFIT_DEBUG:
    print(f'Skipping overfit-debug config rewrite for {PRESET}; using the graduated benchmark config as-is.')
run_script(*make_config_args)
active_config = json.loads(TEMP_CONFIG_PATH.read_text(encoding='utf-8'))
ACTIVE_CONFIG_PATH = TEMP_CONFIG_PATH
ACTIVE_RUN_NAME = active_config.get('debug_run_name', RUN_NAME)
OUTPUT_ROOT = PROJECT_ROOT / active_config['train']['output_dir']
RUN_OUTPUT_DIR = OUTPUT_ROOT / ACTIVE_RUN_NAME
print('ACTIVE_CONFIG_PATH =', ACTIVE_CONFIG_PATH)
print('ACTIVE_RUN_NAME =', ACTIVE_RUN_NAME)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
pprint(active_config)


## Optional Fetch Subset


In [ ]:
if RUN_FETCH:
    run_script('scripts/fetch_region_subset.py', '--config', str(ACTIVE_CONFIG_PATH))


## Preprocess, Train, Export


In [ ]:
RUNS = ['no_edit', 'support_rule', 'consensus', 'target_only', 'full_hybrid', 'full_neural_only']

run_script('scripts/preprocess_dataset.py', '--config', str(ACTIVE_CONFIG_PATH))

if any(run_name in RUNS for run_name in ['no_edit', 'support_rule', 'consensus']):
    run_script('scripts/run_baselines.py', '--config', str(ACTIVE_CONFIG_PATH))

for learned_run in ['target_only', 'full']:
    if learned_run in RUNS or (learned_run == 'full' and any(run in RUNS for run in ['full_hybrid', 'full_neural_only'])):
        run_script('scripts/train_model.py', '--config', str(ACTIVE_CONFIG_PATH), '--run-name', learned_run)

run_script('scripts/export_summary.py', '--config', str(ACTIVE_CONFIG_PATH), '--warn-on-regression-failure')
summary_path = OUTPUT_ROOT / 'benchmark_summary.json'
print('summary_path =', summary_path)
if not summary_path.exists():
    raise FileNotFoundError(f'Expected benchmark summary was not created: {summary_path}')


## Compare Summaries


In [ ]:
summary_path = OUTPUT_ROOT / 'benchmark_summary.json'
if not summary_path.exists():
    print('benchmark_summary.json missing; exporting summary now...')
    run_script('scripts/export_summary.py', '--config', str(ACTIVE_CONFIG_PATH), '--warn-on-regression-failure')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
pprint(summary)

gap_path = OUTPUT_ROOT / 'full' / 'hybrid_gap_summary.json'
if gap_path.exists():
    print('\nHybrid gap summary:')
    pprint(json.loads(gap_path.read_text(encoding='utf-8')))
else:
    print('\nHybrid gap summary missing; train full_hybrid/full_neural_only to create it.')


## Thresholded Decode Probe


In [ ]:
thresholded_reports_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'thresholded',
        '--run-output-dir', str(probe_dir),
        '--example-filters', 'sub_,ins_,del_,copy_',
    )
    thresholded_reports_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'thresholded_probe.json').read_text(encoding='utf-8'))
pprint({name: reports[:2] for name, reports in thresholded_reports_by_checkpoint.items()})


## Direct Argmax-Edit Probe


In [ ]:
if ENABLE_ARGMAX_DEBUG:
    argmax_reports_by_checkpoint = {}
    for checkpoint_name in ['best.ckpt', 'last.ckpt']:
        checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
        probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
        run_script(
            'scripts/debug_probe.py',
            '--config', str(ACTIVE_CONFIG_PATH),
            '--checkpoint', str(checkpoint_path),
            '--mode', 'argmax',
            '--run-output-dir', str(probe_dir),
            '--example-filters', 'sub_,ins_,del_,copy_',
        )
        argmax_reports_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'argmax_probe.json').read_text(encoding='utf-8'))
    pprint({name: reports[:2] for name, reports in argmax_reports_by_checkpoint.items()})


## Per-Class Learnability


In [ ]:
classwise_probe_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'classwise',
        '--run-output-dir', str(probe_dir),
    )
    classwise_probe_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'classwise_probe.json').read_text(encoding='utf-8'))
pprint(classwise_probe_by_checkpoint)


## Missed Hard-Edit Evidence


In [ ]:
missed_evidence_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'missed_evidence',
        '--run-output-dir', str(probe_dir),
    )
    missed_evidence_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'missed_evidence_probe.json').read_text(encoding='utf-8'))
pprint({name: reports[:5] for name, reports in missed_evidence_by_checkpoint.items()})

false_sub_by_checkpoint = {}
ins_payload_by_checkpoint = {}
hybrid_gap_by_checkpoint = {}
hybrid_miss_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'false_sub',
        '--run-output-dir', str(probe_dir),
    )
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'ins_payload',
        '--run-output-dir', str(probe_dir),
    )
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'hybrid_gap',
        '--run-output-dir', str(probe_dir),
    )
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'hybrid_miss',
        '--run-output-dir', str(probe_dir),
    )
    false_sub_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'false_sub_probe.json').read_text(encoding='utf-8'))
    ins_payload_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'ins_payload_probe.json').read_text(encoding='utf-8'))
    hybrid_gap_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'hybrid_gap_probe.json').read_text(encoding='utf-8'))
    hybrid_miss_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'hybrid_miss_probe.json').read_text(encoding='utf-8'))
print('\nFalse SUB diagnostics:')
pprint(false_sub_by_checkpoint)
print('\nINS payload diagnostics:')
pprint(ins_payload_by_checkpoint)
print('\nHybrid gap diagnostics:')
pprint({name: report['summary'] for name, report in hybrid_gap_by_checkpoint.items()})
print('\nHybrid missed-edit diagnostics (SUB_T / boundary INS_A blockers live here):')
pprint(hybrid_miss_by_checkpoint)


## Scale Synthetic Test

This optional cell runs the larger multi-seed synthetic benchmark from inside the notebook. It is intentionally off by default so opening/running the notebook does not accidentally start a heavier experiment.


In [ ]:
RUN_SCALE_SYNTHETIC = True
SCALE_SEEDS = '47,48,49,50,51'
SCALE_BASE_CONFIG = PROJECT_ROOT / 'configs' / 'debug_synthetic_generalization_large.yaml'
SCALE_OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'debug_synthetic_generalization_multiseed'

if RUN_SCALE_SYNTHETIC:
    run_script(
        'scripts/run_synthetic_multiseed.py',
        '--base-config', str(SCALE_BASE_CONFIG),
        '--output-root', str(SCALE_OUTPUT_ROOT),
        '--seeds', SCALE_SEEDS,
    )
    index_path = SCALE_OUTPUT_ROOT / 'multiseed_index.json'
    summary_path = SCALE_OUTPUT_ROOT / 'multiseed_summary.json'
    print('multiseed_index =', index_path)
    print('multiseed_summary =', summary_path)
    if summary_path.exists():
        pprint(json.loads(summary_path.read_text(encoding='utf-8')))
else:
    print('Scale synthetic test is configured but not run.')
    print('Set RUN_SCALE_SYNTHETIC = True to execute it from this notebook.')
    print(PYTHON, 'scripts/run_synthetic_multiseed.py', '--base-config', SCALE_BASE_CONFIG, '--output-root', SCALE_OUTPUT_ROOT, '--seeds', SCALE_SEEDS)


## Mandatory Overfit Gate

The notebook now runs `scripts/run_overfit_suite.py` before benchmark/debug training when `PRESET = 'debug_tiny'`.
That gate proves:

- single-example `SUB` overfit
- single-example `INS` overfit
- single-example `DEL` overfit
- mixed 4-example overfit


In [ ]:
print('ENABLE_OVERFIT_DEBUG =', ENABLE_OVERFIT_DEBUG)
print('OVERFIT_RUN_NAME =', OVERFIT_RUN_NAME)
print('OVERFIT_NUM_EXAMPLES =', OVERFIT_NUM_EXAMPLES)
print('OVERFIT_CASES =', OVERFIT_CASES)
